# Store Sales Forecasting Pipeline

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor

warnings.filterwarnings('ignore')

try:
    from lightgbm import LGBMRegressor
except Exception:
    LGBMRegressor = None

try:
    from xgboost import XGBRegressor
except Exception:
    XGBRegressor = None

INPUT = Path('/kaggle/input/store-sales-time-series-forecasting')
if not INPUT.exists():
    INPUT = Path('data')

OUTPUT = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('.')

## Load Data

In [ ]:
train = pd.read_csv(INPUT / 'train.csv', parse_dates=['date'])
test = pd.read_csv(INPUT / 'test.csv', parse_dates=['date'])
stores = pd.read_csv(INPUT / 'stores.csv')
holidays = pd.read_csv(INPUT / 'holidays_events.csv', parse_dates=['date'])
transactions = pd.read_csv(INPUT / 'transactions.csv', parse_dates=['date'])

train['is_test'] = 0
test['is_test'] = 1
test['sales'] = np.nan

data = pd.concat([train, test], ignore_index=True, sort=False)
data = data.merge(stores[['store_nbr', 'cluster']], on='store_nbr', how='left')
data = data.sort_values(['store_nbr', 'family', 'date']).reset_index(drop=True)

data.shape

## Metrics

In [ ]:
def rmsle(y_true, y_pred):
    y_true = np.asarray(y_true, dtype='float64')
    y_pred = np.clip(np.asarray(y_pred, dtype='float64'), 0, None)
    return float(np.sqrt(np.mean((np.log1p(y_pred) - np.log1p(y_true)) ** 2)))

def mae(y_true, y_pred):
    y_true = np.asarray(y_true, dtype='float64')
    y_pred = np.asarray(y_pred, dtype='float64')
    return float(np.mean(np.abs(y_pred - y_true)))

## Features

In [ ]:
holiday_dates = holidays.copy()
if 'transferred' in holiday_dates.columns:
    holiday_dates = holiday_dates[holiday_dates['transferred'] == False]
holiday_dates = set(holiday_dates['date'].dt.normalize())

data['dayofweek'] = data['date'].dt.dayofweek
data['day'] = data['date'].dt.day
data['month'] = data['date'].dt.month
data['quarter'] = data['date'].dt.quarter
data['year'] = data['date'].dt.year
data['is_weekend'] = (data['dayofweek'] >= 5).astype('int8')
data['is_month_end'] = data['date'].dt.is_month_end.astype('int8')
data['is_holiday'] = data['date'].dt.normalize().isin(holiday_dates).astype('int8')
data['promo_active'] = (data['onpromotion'] > 0).astype('int8')
data['onpromotion_log'] = np.log1p(data['onpromotion'])
data['family_code'] = data['family'].astype('category').cat.codes.astype('int16')
data['log_sales'] = np.log1p(data['sales'].clip(lower=0))

sf = data.groupby(['store_nbr', 'family'], observed=True)['log_sales']
for lag in [16, 28, 56, 112]:
    data[f'sf_lag_{lag}'] = sf.shift(lag)

data['sf_roll_mean_28'] = sf.transform(lambda s: s.shift(16).rolling(28, min_periods=4).mean())
data['sf_roll_mean_56'] = sf.transform(lambda s: s.shift(16).rolling(56, min_periods=8).mean())
data['sf_dow_mean_8'] = data.groupby(['store_nbr', 'family', 'dayofweek'], observed=True)['log_sales'].transform(lambda s: s.shift(1).rolling(8, min_periods=2).mean())
data['cluster_family_dow_mean_8'] = data.groupby(['cluster', 'family', 'dayofweek'], observed=True)['log_sales'].transform(lambda s: s.shift(1).rolling(8, min_periods=2).mean())
data['family_promo_dow_mean_8'] = data.groupby(['family', 'promo_active', 'dayofweek'], observed=True)['log_sales'].transform(lambda s: s.shift(1).rolling(8, min_periods=2).mean())

In [ ]:
store_dates = data[['date', 'store_nbr']].drop_duplicates().sort_values(['store_nbr', 'date'])
store_dates = store_dates.merge(transactions, on=['date', 'store_nbr'], how='left')
store_dates['log_transactions'] = np.log1p(store_dates['transactions'])

sg = store_dates.groupby('store_nbr', observed=True)['log_transactions']
store_dates['txn_lag_16'] = sg.shift(16)
store_dates['txn_roll_mean_28'] = sg.transform(lambda s: s.shift(16).rolling(28, min_periods=4).mean())
store_dates['txn_dow_mean_8'] = store_dates.groupby(['store_nbr', store_dates['date'].dt.dayofweek], observed=True)['log_transactions'].transform(lambda s: s.shift(1).rolling(8, min_periods=2).mean())

data = data.merge(
    store_dates[['date', 'store_nbr', 'txn_lag_16', 'txn_roll_mean_28', 'txn_dow_mean_8']],
    on=['date', 'store_nbr'],
    how='left',
)

feature_cols = [
    'store_nbr', 'family_code', 'cluster', 'dayofweek', 'day', 'month', 'quarter', 'year',
    'is_weekend', 'is_month_end', 'is_holiday', 'onpromotion', 'onpromotion_log', 'promo_active',
    'sf_lag_16', 'sf_lag_28', 'sf_lag_56', 'sf_lag_112', 'sf_roll_mean_28', 'sf_roll_mean_56',
    'sf_dow_mean_8', 'cluster_family_dow_mean_8', 'family_promo_dow_mean_8',
    'txn_lag_16', 'txn_roll_mean_28', 'txn_dow_mean_8',
]

len(feature_cols)

## Validation Split

In [ ]:
valid_start = train['date'].max() - pd.Timedelta(days=15)
train_start = pd.Timestamp('2016-01-01')

train_mask = (data['is_test'] == 0) & (data['date'] < valid_start) & (data['date'] >= train_start)
valid_mask = (data['is_test'] == 0) & (data['date'] >= valid_start)
test_mask = data['is_test'] == 1

X_train = data.loc[train_mask, feature_cols].replace([np.inf, -np.inf], np.nan)
X_valid = data.loc[valid_mask, feature_cols].replace([np.inf, -np.inf], np.nan)
X_test = data.loc[test_mask, feature_cols].replace([np.inf, -np.inf], np.nan)
y_train = data.loc[train_mask, 'log_sales']
y_valid_sales = data.loc[valid_mask, 'sales'].to_numpy()

fill_values = X_train.median(numeric_only=True)
X_train = X_train.fillna(fill_values).fillna(0)
X_valid = X_valid.fillna(fill_values).fillna(0)
X_test = X_test.fillna(fill_values).fillna(0)

X_train.shape, X_valid.shape, X_test.shape

## Train And Compare Models

In [ ]:
def make_models():
    models = {}
    if LGBMRegressor is not None:
        models['lightgbm'] = LGBMRegressor(
            objective='regression',
            n_estimators=700,
            learning_rate=0.05,
            num_leaves=63,
            min_child_samples=120,
            subsample=0.85,
            colsample_bytree=0.85,
            random_state=42,
            n_jobs=-1,
            verbose=-1,
        )
    if XGBRegressor is not None:
        models['xgboost'] = XGBRegressor(
            objective='reg:squarederror',
            n_estimators=700,
            learning_rate=0.05,
            max_depth=8,
            min_child_weight=20,
            subsample=0.85,
            colsample_bytree=0.85,
            tree_method='hist',
            random_state=42,
            n_jobs=-1,
        )
    if not models:
        models['hist_gradient_boosting'] = HistGradientBoostingRegressor(
            max_iter=350,
            learning_rate=0.05,
            l2_regularization=0.05,
            random_state=42,
        )
    return models

models = make_models()
valid_predictions = {}
score_rows = []

for name, model in models.items():
    model.fit(X_train, y_train)
    pred = np.expm1(model.predict(X_valid))
    pred = np.clip(pred, 0, None)
    valid_predictions[name] = pred
    score_rows.append({'model': name, 'rmsle': rmsle(y_valid_sales, pred), 'mae': mae(y_valid_sales, pred)})

if {'lightgbm', 'xgboost'}.issubset(valid_predictions):
    pred = 0.7 * valid_predictions['lightgbm'] + 0.3 * valid_predictions['xgboost']
    valid_predictions['weighted_ensemble_70_30'] = pred
    score_rows.append({'model': 'weighted_ensemble_70_30', 'rmsle': rmsle(y_valid_sales, pred), 'mae': mae(y_valid_sales, pred)})
elif len(valid_predictions) > 1:
    pred = np.mean(list(valid_predictions.values()), axis=0)
    valid_predictions['average_ensemble'] = pred
    score_rows.append({'model': 'average_ensemble', 'rmsle': rmsle(y_valid_sales, pred), 'mae': mae(y_valid_sales, pred)})

scores = pd.DataFrame(score_rows).sort_values('rmsle').reset_index(drop=True)
scores

## Train Final Models

In [ ]:
final_mask = (data['is_test'] == 0) & (data['date'] >= train_start)
X_final = data.loc[final_mask, feature_cols].replace([np.inf, -np.inf], np.nan)
y_final = data.loc[final_mask, 'log_sales']

final_fill_values = X_final.median(numeric_only=True)
X_final = X_final.fillna(final_fill_values).fillna(0)
X_test_final = X_test.fillna(final_fill_values).fillna(0)

final_models = make_models()
test_predictions = {}

for name, model in final_models.items():
    model.fit(X_final, y_final)
    pred = np.expm1(model.predict(X_test_final))
    test_predictions[name] = np.clip(pred, 0, None)

if {'lightgbm', 'xgboost'}.issubset(test_predictions):
    final_pred = 0.7 * test_predictions['lightgbm'] + 0.3 * test_predictions['xgboost']
else:
    final_pred = np.mean(list(test_predictions.values()), axis=0)

final_pred = np.clip(final_pred, 0, None)
final_pred[:10]

## Submission

In [ ]:
submission = test[['id']].copy()
submission['sales'] = final_pred
submission.to_csv(OUTPUT / 'submission.csv', index=False)

submission.head(), submission.shape, OUTPUT / 'submission.csv'